# Data Lake - Olimpiadas\n\nConstrucao do Data Lake local seguindo a arquitetura medallion (raw, bronze, gold) com dados olimpicos historicos e Paris 2024.

In [1]:
import pandas as pd
import json
import shutil
import os
from pathlib import Path
from datetime import datetime
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
BASE_DIR = Path("data_lake")
RAW_DIR = BASE_DIR / "raw"
BRONZE_DIR = BASE_DIR / "bronze"
GOLD_DIR = BASE_DIR / "gold"

SOURCE_HISTORICO = "dados-historicos/world_olympedia_olympics_athlete_event_result.csv"
SOURCE_PARIS = "dados-paris-2024/medallists.csv"

for d in [RAW_DIR, BRONZE_DIR, GOLD_DIR / "analise_medalhas", GOLD_DIR / "analise_modalidades", GOLD_DIR / "analise_genero"]:
    d.mkdir(parents=True, exist_ok=True)


def criar_metadados(nome, fonte, descricao, colunas, qtd_linhas, formato, observacoes=""):
    return {
        "nome": nome,
        "fonte": fonte,
        "descricao": descricao,
        "formato": formato,
        "colunas": colunas,
        "quantidade_linhas": qtd_linhas,
        "data_coleta": datetime.now().strftime("%Y-%m-%d"),
        "observacoes": observacoes
    }


def salvar_metadados(metadados, caminho):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=4)

## Camada Raw\n\nCopia dos arquivos originais (CSV) para a pasta raw, junto com seus metadados em JSON.

In [3]:
shutil.copy2(SOURCE_HISTORICO, RAW_DIR / "olympics_historico.csv")
df_hist = pd.read_csv(RAW_DIR / "olympics_historico.csv")

meta_hist = criar_metadados(
    nome="Olympics Historico",
    fonte="Kaggle - World Olympedia",
    descricao="Dados de atletas e resultados de todas as edicoes dos Jogos Olimpicos",
    colunas=[{"nome": col, "tipo": str(df_hist[col].dtype)} for col in df_hist.columns],
    qtd_linhas=len(df_hist),
    formato="csv",
    observacoes="Contem dados desde 1896 ate 2022, incluindo esportes de verao e inverno"
)
salvar_metadados(meta_hist, RAW_DIR / "olympics_historico.json")

print(f"Historico: {len(df_hist)} linhas, {len(df_hist.columns)} colunas")
df_hist.head()

/tmp/ipykernel_2172/1868076567.py:2: DtypeWarning: Columns (0: medal) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hist = pd.read_csv(RAW_DIR / "olympics_historico.csv")


Historico: 316834 linhas, 11 colunas


,edition,edition_id,country_noc,sport,event,result_id,athlete,athlete_id,position,medal,is_team_sport
0,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Willy von Eschen,98710,Did not finish,NaN,False
1,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,"Jean, Comte de Beaumont",42118,Did not start,NaN,False
2,1928 Winter Olympics,30,FRA,Skeleton,"Skeleton, Men",1,Pierre Dormeuil,85267,Did not finish,NaN,False
3,1928 Winter Olympics,30,GBR,Skeleton,"Skeleton, Men",1,Lord Brabazon of Tara,1202561,Did not start,NaN,False
4,1928 Winter Olympics,30,SUI,Skeleton,"Skeleton, Men",1,Alexander Berner,84063,5,NaN,False


In [4]:
shutil.copy2(SOURCE_PARIS, RAW_DIR / "olympics_paris2024.csv")
df_paris = pd.read_csv(RAW_DIR / "olympics_paris2024.csv")

meta_paris = criar_metadados(
    nome="Olympics Paris 2024",
    fonte="Kaggle - Paris 2024 Olympic Games",
    descricao="Dados dos medalhistas dos Jogos Olimpicos de Paris 2024",
    colunas=[{"nome": col, "tipo": str(df_paris[col].dtype)} for col in df_paris.columns],
    qtd_linhas=len(df_paris),
    formato="csv",
    observacoes="Dados oficiais das Olimpiadas de Paris 2024"
)
salvar_metadados(meta_paris, RAW_DIR / "olympics_paris2024.json")

print(f"Paris 2024: {len(df_paris)} linhas, {len(df_paris.columns)} colunas")
df_paris.head()

Paris 2024: 2315 linhas, 21 colunas


,medal_date,medal_type,medal_code,name,gender,country_code,country,country_long,nationality_code,nationality,...,team,team_gender,discipline,event,event_type,url_event,birth_date,code_athlete,code_team,is_medallist
0,2024-07-27,Gold Medal,1.0,EVENEPOEL Remco,Male,BEL,Belgium,Belgium,BEL,Belgium,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,2000-01-25,1903136,NaN,True
1,2024-07-27,Silver Medal,2.0,GANNA Filippo,Male,ITA,Italy,Italy,ITA,Italy,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1996-07-25,1923520,NaN,True
2,2024-07-27,Bronze Medal,3.0,van AERT Wout,Male,BEL,Belgium,Belgium,BEL,Belgium,...,NaN,NaN,Cycling Road,Men's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/men-s-indi...,1994-09-15,1903147,NaN,True
3,2024-07-27,Gold Medal,1.0,BROWN Grace,Female,AUS,Australia,Australia,AUS,Australia,...,NaN,NaN,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1992-07-07,1940173,NaN,True
4,2024-07-27,Silver Medal,2.0,HENDERSON Anna,Female,GBR,Great Britain,Great Britain,GBR,Great Britain,...,NaN,NaN,Cycling Road,Women's Individual Time Trial,ATH,/en/paris-2024/results/cycling-road/women-s-in...,1998-11-14,1912525,NaN,True


## Camada Bronze\n\nConversao dos CSVs para Parquet e integracao entre as fontes de dados.

In [5]:
df_hist.to_parquet(BRONZE_DIR / "olympics_historico.parquet", engine="pyarrow", index=False)

meta_hist_parquet = criar_metadados(
    nome="Olympics Historico",
    fonte="Camada raw - olympics_historico.csv",
    descricao="Dados historicos convertidos para formato Parquet",
    colunas=[{"nome": col, "tipo": str(df_hist[col].dtype)} for col in df_hist.columns],
    qtd_linhas=len(df_hist),
    formato="parquet"
)
salvar_metadados(meta_hist_parquet, BRONZE_DIR / "olympics_historico.json")

df_paris.to_parquet(BRONZE_DIR / "olympics_paris2024.parquet", engine="pyarrow", index=False)

meta_paris_parquet = criar_metadados(
    nome="Olympics Paris 2024",
    fonte="Camada raw - olympics_paris2024.csv",
    descricao="Dados de Paris 2024 convertidos para formato Parquet",
    colunas=[{"nome": col, "tipo": str(df_paris[col].dtype)} for col in df_paris.columns],
    qtd_linhas=len(df_paris),
    formato="parquet"
)
salvar_metadados(meta_paris_parquet, BRONZE_DIR / "olympics_paris2024.json")

print("Parquet files criados com sucesso")

Parquet files criados com sucesso


### Integracao: Medalhas 1896-2024\n\nJoin entre os dados historicos e Paris 2024, filtrando apenas medalhistas.

In [6]:
medalhas_hist = df_hist[df_hist["medal"].isin(["Gold", "Silver", "Bronze"])].copy()
medalhas_hist["ano"] = medalhas_hist["edition"].str[:4].astype(int)
medalhas_hist = medalhas_hist.rename(columns={"country_noc": "pais", "athlete": "atleta", "medal": "medalha"})
medalhas_hist = medalhas_hist[["ano", "edition", "pais", "sport", "event", "atleta", "medalha"]]

medalhas_paris = df_paris.copy()
medalhas_paris["medalha"] = medalhas_paris["medal_type"].str.replace(" Medal", "")
medalhas_paris["ano"] = 2024
medalhas_paris["edition"] = "2024 Summer Olympics"
medalhas_paris = medalhas_paris.rename(columns={"country_code": "pais", "name": "atleta", "discipline": "sport"})
medalhas_paris = medalhas_paris[["ano", "edition", "pais", "sport", "event", "atleta", "medalha"]]

df_medalhas = pd.concat([medalhas_hist, medalhas_paris], ignore_index=True)

df_medalhas.to_parquet(BRONZE_DIR / "medalhas_1896_2024.parquet", engine="pyarrow", index=False)
df_medalhas.to_csv(BRONZE_DIR / "medalhas_1896_2024.csv", index=False)

meta_medalhas = criar_metadados(
    nome="Medalhas 1896-2024",
    fonte="Integracao olympics_historico + olympics_paris2024",
    descricao="Dataset unificado de todos os medalhistas olimpicos de 1896 a 2024",
    colunas=[{"nome": col, "tipo": str(df_medalhas[col].dtype)} for col in df_medalhas.columns],
    qtd_linhas=len(df_medalhas),
    formato="parquet/csv",
    observacoes="Join entre dados historicos e Paris 2024, apenas atletas medalhistas"
)
salvar_metadados(meta_medalhas, BRONZE_DIR / "medalhas_1896_2024.json")

print(f"Medalhas unificadas: {len(df_medalhas)} registros")
df_medalhas.head()

Medalhas unificadas: 47002 registros


,ano,edition,pais,sport,event,atleta,medalha
0,1928,1928 Winter Olympics,USA,Skeleton,"Skeleton, Men",Jennison Heaton,Gold
1,1948,1948 Winter Olympics,ITA,Skeleton,"Skeleton, Men",Nino Bibbia,Gold
2,2002,2002 Winter Olympics,USA,Skeleton,"Skeleton, Men","Jim Shea, Jr.",Gold
3,2002,2002 Winter Olympics,USA,Skeleton,"Skeleton, Women",Tristan Gale,Gold
4,2006,2006 Winter Olympics,CAN,Skeleton,"Skeleton, Men",Duff Gibson,Gold


### Integracao: Modalidades 1896-2024\n\nContagem de medalhas por esporte e edicao.

In [7]:
df_modalidades = df_medalhas.pivot_table(
    index=["ano", "edition", "sport"],
    columns="medalha",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()

for col in ["Gold", "Silver", "Bronze"]:
    if col not in df_modalidades.columns:
        df_modalidades[col] = 0

df_modalidades = df_modalidades[["ano", "edition", "sport", "Gold", "Silver", "Bronze"]]
df_modalidades["Total"] = df_modalidades[["Gold", "Silver", "Bronze"]].sum(axis=1)
df_modalidades.columns.name = None

df_modalidades.to_parquet(BRONZE_DIR / "modalidades_1896_2024.parquet", engine="pyarrow", index=False)
df_modalidades.to_csv(BRONZE_DIR / "modalidades_1896_2024.csv", index=False)

meta_mod = criar_metadados(
    nome="Modalidades 1896-2024",
    fonte="Integracao medalhas_1896_2024",
    descricao="Contagem de medalhas por esporte e edicao olimpica",
    colunas=[{"nome": col, "tipo": str(df_modalidades[col].dtype)} for col in df_modalidades.columns],
    qtd_linhas=len(df_modalidades),
    formato="parquet/csv"
)
salvar_metadados(meta_mod, BRONZE_DIR / "modalidades_1896_2024.json")

print(f"Modalidades: {len(df_modalidades)} registros")
df_modalidades.head()

Modalidades: 1116 registros


,ano,edition,sport,Gold,Silver,Bronze,Total
0,1896,1896 Summer Olympics,Artistic Gymnastics,26,9,6,41
1,1896,1896 Summer Olympics,Athletics,12,13,12,37
2,1896,1896 Summer Olympics,Cycling Road,1,1,1,3
3,1896,1896 Summer Olympics,Cycling Track,5,5,3,13
4,1896,1896 Summer Olympics,Fencing,3,3,3,9


### Integracao: Atletas por Sexo\n\nClassificacao de medalhistas por genero ao longo das edicoes.

In [8]:
def extrair_genero_evento(evento):
    evento = str(evento).lower()
    if "women" in evento or "female" in evento:
        return "Female"
    elif "men" in evento or "male" in evento:
        return "Male"
    return "Mixed"

atletas_hist = medalhas_hist.copy()
atletas_hist["genero"] = df_hist.loc[df_hist["medal"].isin(["Gold", "Silver", "Bronze"]), "event"].apply(extrair_genero_evento).values
atletas_hist = atletas_hist[["ano", "edition", "pais", "sport", "atleta", "medalha", "genero"]]

atletas_paris = medalhas_paris.copy()
atletas_paris["genero"] = df_paris["gender"].values
atletas_paris = atletas_paris[["ano", "edition", "pais", "sport", "atleta", "medalha", "genero"]]

df_atletas_sexo = pd.concat([atletas_hist, atletas_paris], ignore_index=True)

df_atletas_sexo.to_parquet(BRONZE_DIR / "atletas_por_sexo.parquet", engine="pyarrow", index=False)
df_atletas_sexo.to_csv(BRONZE_DIR / "atletas_por_sexo.csv", index=False)

meta_sexo = criar_metadados(
    nome="Atletas por Sexo",
    fonte="Integracao olympics_historico + olympics_paris2024",
    descricao="Medalhistas classificados por genero de 1896 a 2024",
    colunas=[{"nome": col, "tipo": str(df_atletas_sexo[col].dtype)} for col in df_atletas_sexo.columns],
    qtd_linhas=len(df_atletas_sexo),
    formato="parquet/csv",
    observacoes="Genero extraido do nome do evento (historico) ou coluna gender (Paris 2024)"
)
salvar_metadados(meta_sexo, BRONZE_DIR / "atletas_por_sexo.json")

print(f"Atletas por sexo: {len(df_atletas_sexo)} registros")
df_atletas_sexo["genero"].value_counts()

Atletas por sexo: 47002 registros


genero
Male      30091
Female    13992
Mixed      2919
Name: count, dtype: int64

## Camada Gold\n\nAnalises finais e visualizacoes a partir dos dados integrados.

### Analise de Medalhas - Quadro de medalhas por pais

In [9]:
quadro_medalhas = df_medalhas.pivot_table(
    index="pais",
    columns="medalha",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()

for col in ["Gold", "Silver", "Bronze"]:
    if col not in quadro_medalhas.columns:
        quadro_medalhas[col] = 0

quadro_medalhas = quadro_medalhas[["pais", "Gold", "Silver", "Bronze"]]
quadro_medalhas["Total"] = quadro_medalhas[["Gold", "Silver", "Bronze"]].sum(axis=1)
quadro_medalhas.columns.name = None
quadro_medalhas = quadro_medalhas.sort_values(["Gold", "Silver", "Bronze"], ascending=False).reset_index(drop=True)

gold_medalhas_dir = GOLD_DIR / "analise_medalhas"
quadro_medalhas.to_csv(gold_medalhas_dir / "medalhas_summary.csv", index=False)

print(f"Quadro de medalhas: {len(quadro_medalhas)} paises")
quadro_medalhas.head(15)

Quadro de medalhas: 160 paises


,pais,Gold,Silver,Bronze,Total
0,USA,3076,1956,1592,6624
1,URS,1099,743,704,2546
2,GBR,852,836,846,2534
3,GER,844,832,881,2557
4,ITA,628,600,611,1839
5,FRA,622,782,759,2163
6,CAN,596,545,591,1732
7,SWE,529,586,557,1672
8,CHN,495,486,385,1366
9,HUN,466,356,427,1249


In [10]:
top15 = quadro_medalhas.head(15).set_index("pais")

fig, ax = plt.subplots(figsize=(12, 6))
top15[["Gold", "Silver", "Bronze"]].plot(
    kind="barh",
    stacked=True,
    color=["#FFD700", "#C0C0C0", "#CD7F32"],
    ax=ax
)
ax.set_xlabel("Quantidade de Medalhas")
ax.set_ylabel("")
ax.set_title("Top 15 Paises - Quadro de Medalhas (1896-2024)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(gold_medalhas_dir / "medalhas_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [11]:
meta_gold_medalhas = criar_metadados(
    nome="Analise de Medalhas",
    fonte="Camada bronze - medalhas_1896_2024",
    descricao="Quadro de medalhas por pais de 1896 a 2024, com visualizacao dos top 15",
    colunas=[{"nome": col, "tipo": str(quadro_medalhas[col].dtype)} for col in quadro_medalhas.columns],
    qtd_linhas=len(quadro_medalhas),
    formato="csv/png",
    observacoes="Contagem de medalhas individuais (cada atleta conta como uma medalha)"
)
salvar_metadados(meta_gold_medalhas, gold_medalhas_dir / "metadata.json")

### Analise de Modalidades - Esportes com mais medalhas

In [12]:
gold_mod_dir = GOLD_DIR / "analise_modalidades"

modalidades_summary = df_modalidades.groupby("sport")[["Gold", "Silver", "Bronze", "Total"]].sum()
modalidades_summary = modalidades_summary.sort_values("Total", ascending=False).reset_index()

modalidades_summary.to_csv(gold_mod_dir / "modalidades_summary.csv", index=False)

print(f"Total de esportes: {len(modalidades_summary)}")
modalidades_summary.head(15)

Total de esportes: 85


,sport,Gold,Silver,Bronze,Total
0,Athletics,1606,1586,1548,4740
1,Swimming,1247,1141,1109,3497
2,Rowing,1077,1061,1080,3218
3,Artistic Gymnastics,810,774,759,2343
4,Football,742,707,739,2188
5,Fencing,649,643,630,1922
6,Hockey,627,624,632,1883
7,Ice Hockey,622,628,624,1874
8,Wrestling,446,446,536,1428
9,Sailing,479,445,396,1320


In [13]:
top15_esportes = modalidades_summary.head(15).set_index("sport")

fig, ax = plt.subplots(figsize=(12, 6))
top15_esportes[["Gold", "Silver", "Bronze"]].plot(
    kind="barh",
    stacked=True,
    color=["#FFD700", "#C0C0C0", "#CD7F32"],
    ax=ax
)
ax.set_xlabel("Quantidade de Medalhas")
ax.set_ylabel("")
ax.set_title("Top 15 Esportes por Medalhas (1896-2024)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(gold_mod_dir / "modalidades_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [14]:
meta_gold_mod = criar_metadados(
    nome="Analise de Modalidades",
    fonte="Camada bronze - modalidades_1896_2024",
    descricao="Ranking dos esportes olimpicos por quantidade de medalhas distribuidas",
    colunas=[{"nome": col, "tipo": str(modalidades_summary[col].dtype)} for col in modalidades_summary.columns],
    qtd_linhas=len(modalidades_summary),
    formato="csv/png"
)
salvar_metadados(meta_gold_mod, gold_mod_dir / "metadata.json")

### Analise de Genero - Evolucao da participacao por sexo

In [15]:
gold_genero_dir = GOLD_DIR / "analise_genero"

genero_por_ano = df_atletas_sexo.pivot_table(
    index="ano",
    columns="genero",
    values="atleta",
    aggfunc="count",
    fill_value=0
).reset_index()
genero_por_ano.columns.name = None

if "Male" not in genero_por_ano.columns:
    genero_por_ano["Male"] = 0
if "Female" not in genero_por_ano.columns:
    genero_por_ano["Female"] = 0

genero_por_ano["Total"] = genero_por_ano.select_dtypes(include="number").sum(axis=1)
genero_por_ano["Percentual_Feminino"] = (genero_por_ano["Female"] / genero_por_ano["Total"] * 100).round(1)

genero_por_ano.to_csv(gold_genero_dir / "genero_summary.csv", index=False)

print(f"Edicoes analisadas: {len(genero_por_ano)}")
genero_por_ano.tail(10)

Edicoes analisadas: 38


,ano,Female,Male,Mixed,Total,Percentual_Feminino
28,2006,228,289,18,2541,9.0
29,2008,920,1094,66,4088,22.5
30,2010,231,285,18,2544,9.1
31,2012,916,1010,57,3995,22.9
32,2014,246,300,69,2629,9.4
33,2016,962,1038,63,4079,23.6
34,2018,256,294,123,2691,9.5
35,2020,1113,1132,189,4454,25.0
36,2022,265,297,164,2748,9.6
37,2024,1162,1153,0,4339,26.8


In [16]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

genero_por_ano.plot(x="ano", y=["Male", "Female"], kind="area", stacked=True,
                    color=["#4A90D9", "#E74C6F"], alpha=0.7, ax=ax1)
ax1.set_title("Medalhistas por Genero ao Longo do Tempo")
ax1.set_xlabel("Ano")
ax1.set_ylabel("Quantidade")

ax2.plot(genero_por_ano["ano"], genero_por_ano["Percentual_Feminino"], color="#E74C6F", linewidth=2, marker="o", markersize=3)
ax2.set_title("Percentual de Medalhistas Femininas (%)")
ax2.set_xlabel("Ano")
ax2.set_ylabel("Percentual (%)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(gold_genero_dir / "genero_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [17]:
meta_gold_genero = criar_metadados(
    nome="Analise de Genero",
    fonte="Camada bronze - atletas_por_sexo",
    descricao="Evolucao da participacao de medalhistas por genero ao longo das edicoes olimpicas",
    colunas=[{"nome": col, "tipo": str(genero_por_ano[col].dtype)} for col in genero_por_ano.columns],
    qtd_linhas=len(genero_por_ano),
    formato="csv/png"
)
salvar_metadados(meta_gold_genero, gold_genero_dir / "metadata.json")

## Metadata Schema e Resumo Final

In [18]:
schema = {
    "schema_version": "1.0",
    "descricao": "Schema padrao para os arquivos de metadados do Data Lake",
    "campos_obrigatorios": ["nome", "fonte", "descricao", "formato", "colunas", "quantidade_linhas", "data_coleta"],
    "tipos_campos": {
        "nome": "string - nome do dataset",
        "fonte": "string - origem dos dados",
        "descricao": "string - descricao do conteudo",
        "formato": "string - formato do arquivo (csv, parquet, png)",
        "colunas": "array - lista de colunas com nome e tipo",
        "quantidade_linhas": "integer - numero de linhas no dataset",
        "data_coleta": "string - data de criacao (YYYY-MM-DD)",
        "observacoes": "string - informacoes adicionais (opcional)"
    }
}

salvar_metadados(schema, BASE_DIR / "metadata_schema.json")
print("metadata_schema.json criado")

metadata_schema.json criado


In [19]:
for dirpath, dirnames, filenames in sorted(os.walk(BASE_DIR)):
    nivel = dirpath.replace(str(BASE_DIR), "").count(os.sep)
    indent = "  " * nivel
    pasta = os.path.basename(dirpath)
    print(f"{indent}{pasta}/")
    for f in sorted(filenames):
        print(f"{indent}  {f}")

data_lake/
  metadata_schema.json
  bronze/
    atletas_por_sexo.csv
    atletas_por_sexo.json
    atletas_por_sexo.parquet
    medalhas_1896_2024.csv
    medalhas_1896_2024.json
    medalhas_1896_2024.parquet
    modalidades_1896_2024.csv
    modalidades_1896_2024.json
    modalidades_1896_2024.parquet
    olympics_historico.json
    olympics_historico.parquet
    olympics_paris2024.json
    olympics_paris2024.parquet
  gold/
    analise_genero/
      genero_plot.png
      genero_summary.csv
      metadata.json
    analise_medalhas/
      medalhas_plot.png
      medalhas_summary.csv
      metadata.json
    analise_modalidades/
      metadata.json
      modalidades_plot.png
      modalidades_summary.csv
  raw/
    olympics_historico.csv
    olympics_historico.json
    olympics_paris2024.csv
    olympics_paris2024.json
